In [1]:
import pyreadstat
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import flwr as fl
from flwr.simulation import start_simulation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import inspect
import random
import copy


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)



In [3]:
# Data Extraction

path = r"c:\Users\usher\Downloads\NGIR7BDT\NGIR7BFL.DTA"

df, meta = pyreadstat.read_dta(path)

print("Dataset loaded sucessfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded sucessfully
Rows: 41821
Columns: 5394


In [4]:
print("First 30 variables:")
print(df.columns[:30].tolist())

print("\nLast 30 variables:")
print(df.columns[-30:].tolist())

First 30 variables:
['caseid', 'v000', 'v001', 'v002', 'v003', 'v004', 'v005', 'v006', 'v007', 'v008', 'v008a', 'v009', 'v010', 'v011', 'v012', 'v013', 'v014', 'v015', 'v016', 'v017', 'v018', 'v019', 'v019a', 'v020', 'v021', 'v022', 'v023', 'v024', 'v025', 'v026']

Last 30 variables:
['s434ig_1', 's434ig_2', 's434ig_3', 's434ig_4', 's434ig_5', 's434ig_6', 's434ix_1', 's434ix_2', 's434ix_3', 's434ix_4', 's434ix_5', 's434ix_6', 's434iz_1', 's434iz_2', 's434iz_3', 's434iz_4', 's434iz_5', 's434iz_6', 's434k_1', 's434k_2', 's434k_3', 's434k_4', 's434k_5', 's434k_6', 's434l_1', 's434l_2', 's434l_3', 's434l_4', 's434l_5', 's434l_6']


In [5]:
df.head(10)

,caseid,v000,v001,v002,v003,v004,v005,v006,v007,v008,...,s434k_3,s434k_4,s434k_5,s434k_6,s434l_1,s434l_2,s434l_3,s434l_4,s434l_5,s434l_6
0,1 1 2,NG7,1,1,2,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1 6 4,NG7,1,6,4,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1 11 1,NG7,1,11,1,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1 25 2,NG7,1,25,2,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1 30 1,NG7,1,30,1,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1 35 2,NG7,1,35,2,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1 40 3,NG7,1,40,3,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1 40 5,NG7,1,40,5,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1 45 2,NG7,1,45,2,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1 54 1,NG7,1,54,1,1,1335530,9,2018,1425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
print(df['v024'].value_counts().sort_index())
print(meta.variable_value_labels.get("v024"))

v024
1     7772
2     7639
3    10129
4     5571
5     5080
6     5630
Name: count, dtype: int64
{1: 'north central', 2: 'north east', 3: 'north west', 4: 'south east', 5: 'south south', 6: 'south west'}


In [7]:
# finding the possible maternal-health variables

keywords = [
    "birth",
    "pregnancy",
    "maternal",
    "antenatal",
    "delivery",
    "health",
    "faculty",
    "skilled",
    "contraceptive"
]

label_matches = []

for col, label in meta.column_names_to_labels.items():
    label_text = str(label).lower()

    if any(keyword in label_text for keyword in keywords):
        label_matches.append((col, label))

print("Number of matching variables:", len(label_matches))

for col, label in label_matches:
    print(f"{col} - {label}")

Number of matching variables: 1283
v009 - respondent's month of birth
v010 - respondent's year of birth
v011 - date of birth (cmc)
bidx_01 - birth column number
bidx_02 - birth column number
bidx_03 - birth column number
bidx_04 - birth column number
bidx_05 - birth column number
bidx_06 - birth column number
bidx_07 - birth column number
bidx_08 - birth column number
bidx_09 - birth column number
bidx_10 - birth column number
bidx_11 - birth column number
bidx_12 - birth column number
bidx_13 - birth column number
bidx_14 - birth column number
bidx_15 - birth column number
bidx_16 - birth column number
bidx_17 - birth column number
bidx_18 - birth column number
bidx_19 - birth column number
bidx_20 - birth column number
bord_01 - birth order number
bord_02 - birth order number
bord_03 - birth order number
bord_04 - birth order number
bord_05 - birth order number
bord_06 - birth order number
bord_07 - birth order number
bord_08 - birth order number
bord_09 - birth order number
bord_10 

In [8]:
# Identifying the exact variables available

candidate_vars = [
    "v024",  # Region
    "v025",  # Urban/rural
    "v012",  # Age
    "v106",  # Education
    "v190",  # Wealth index
    "v714",  # currently working
    "v130",  # Ethnicity
    "v131",  # Religion
    "v151",  # Sex of household head
    "v481",  # Health insurance
]

for var in candidate_vars:
    if var in df.columns:
        print(f"{var} - {meta.column_names_to_labels.get(var)}")
    else:
        print(f"{var} - NOT FOUND")

v024 - region
v025 - type of place of residence
v012 - respondent's current age
v106 - highest educational level
v190 - wealth index combined
v714 - respondent currently working
v130 - religion
v131 - ethnicity
v151 - sex of household head
v481 - covered by health insurance


In [9]:
df[candidate_vars].head()

,v024,v025,v012,v106,v190,v714,v130,v131,v151,v481
0,1,1,40,3,5,1,2,6,1,0
1,1,1,16,2,5,0,2,9,1,0
2,1,1,37,3,5,0,1,9,2,0
3,1,1,27,3,5,1,2,96,1,0
4,1,1,29,2,5,1,2,96,2,0


In [10]:
# checking the quallity of the variables

candidate_vars = [
    "v024", "v025", "v012", "v106", 'v190',
    "v714", "v130", "v131", "v151", "v481"
]

missing = df[candidate_vars].isna().sum()

print("Missing values:", missing)

print("\nMissing percentage:")
print((missing / len(df) * 100).round(2))

Missing values: v024    0
v025    0
v012    0
v106    0
v190    0
v714    0
v130    0
v131    0
v151    0
v481    0
dtype: int64

Missing percentage:
v024    0.0
v025    0.0
v012    0.0
v106    0.0
v190    0.0
v714    0.0
v130    0.0
v131    0.0
v151    0.0
v481    0.0
dtype: float64


In [11]:
# Understanding the value labels for these variables

for var in candidate_vars:
    print(f"\n{'='*50}")
    print(f"{var} - {meta.column_names_to_labels.get(var)}")
    print(meta.variable_value_labels.get(var, "No value labels"))


v024 - region
{1: 'north central', 2: 'north east', 3: 'north west', 4: 'south east', 5: 'south south', 6: 'south west'}

v025 - type of place of residence
{1: 'urban', 2: 'rural'}

v012 - respondent's current age
No value labels

v106 - highest educational level
{0: 'no education', 1: 'primary', 2: 'secondary', 3: 'higher'}

v190 - wealth index combined
{1: 'poorest', 2: 'poorer', 3: 'middle', 4: 'richer', 5: 'richest'}

v714 - respondent currently working
{0: 'no', 1: 'yes'}

v130 - religion
{1: 'catholic', 2: 'other christian', 3: 'islam', 4: 'traditionalist', 96: 'other'}

v131 - ethnicity
{1: 'ekoi', 2: 'fulani', 3: 'hausa', 4: 'ibibio', 5: 'igala', 6: 'igbo', 7: 'ijaw/izon', 8: 'kanuri/beriberi', 9: 'tiv', 10: 'yoruba', 96: 'other', 98: "don't know"}

v151 - sex of household head
{1: 'male', 2: 'female'}

v481 - covered by health insurance
{0: 'no', 1: 'yes'}


In [12]:
# investigating the actual maternal-health outcomes

outcome_candidates = [
    "m15_1",  # Place of delivery
    "m14_1",  # number of antenatal visits
    "m13_1",  # timing of first antenatal visit
    "m17_1",  # Caesarean section
]

for var in outcome_candidates:
    print("\n" + "="*50)
    print(f"{var} - {meta.column_names_to_labels.get(var)}")
    print("Values Labels:")
    print(meta.variable_value_labels.get(var, "No value labels"))

    print("\nValue counts:")
    print(df[var].value_counts(dropna=False).sort_index())


m15_1 - place of delivery
Values Labels:
{10: 'home', 11: "respondent's home", 12: 'other home', 20: 'public sector', 21: 'government hospital', 22: 'government health center', 23: 'government health post', 26: 'other public sector', 30: 'private sector', 31: 'private hospital/clinic', 36: 'other private sector', 96: 'other'}

Value counts:
m15_1
11     11267
12      1174
21      2972
22      3081
23       243
26        13
31      2647
36        45
96       350
NaN    20029
Name: count, dtype: int64

m14_1 - number of antenatal visits during pregnancy
Values Labels:
{0: 'no antenatal visits', 98: "don't know"}

Value counts:
m14_1
0       5365
1        583
2        968
3       2242
4       2716
5       2344
6       2095
7       1055
8       1079
9        443
10       806
11       160
12       472
13        80
14       113
15       231
16       166
17        56
18       115
19        33
20       343
98       327
NaN    20029
Name: count, dtype: int64

m13_1 - timing of 1st antenatal ch

In [13]:
## Creating a temporary binary version of m15_1 (place of delivery)
# Create a temporary binary facility-delivery outcome
# 0 = home delivery
# 1 = facility delivery

facility_delivery = df["m15_1"].map(
    lambda x: 0 if x in [10, 11, 12]
    else 1 if x in [20, 21, 22, 23, 24, 26, 30, 31, 36]
    else None
)

print("Overall outcome distribution:")
print(facility_delivery.value_counts(dropna=False))

Overall outcome distribution:
m15_1
NaN    20379
0.0    12441
1.0     9001
Name: count, dtype: int64


In [14]:
print("\nFacility delivery by region:")

regional_table = pd.crosstab(
    df["v024"],
    facility_delivery,
    normalize="index"
) * 100

regional_table.index = [
    "North Central",
    "North East",
    "North West",
    "South East",
    "South South",
    "South West"
]

regional_table.columns = ["Home//Non-Facility (%)", "Facility (%)"]

print(regional_table.round(2))


Facility delivery by region:
               Home//Non-Facility (%)  Facility (%)
North Central                   47.97         52.03
North East                      74.01         25.99
North West                      84.32         15.68
South East                      19.29         80.71
South South                     51.32         48.68
South West                      18.66         81.34


In [15]:
df = df.dropna(subset=["m15_1"])

In [16]:
print("Births in last 5 years:")
print(df["v208"].value_counts(dropna=False).sort_index())

Births in last 5 years:
v208
1    11363
2     8852
3     1461
4      108
5        6
6        2
Name: count, dtype: int64


In [17]:
print("m15_1 by v208:")
print(
    pd.crosstab(
        df["v208"],
        df["m15_1"],
        dropna=False
    )
)

m15_1 by v208:
m15_1    11   12    21    22   23  26    31  36   96
v208                                                
1      5255  679  1775  1751  142   7  1524  22  208
2      5145  413  1022  1118   82   6   923  19  124
3       806   76   160   198   17   0   183   4   17
4        58    5    12    13    2   0    17   0    1
5         3    1     2     0    0   0     0   0    0
6         0    0     1     1    0   0     0   0    0


In [18]:
## Feature Selection

feature_candidates = [
    "v012",   # Age
    "v025",   # Urban/rural
    "v106",   # Education
    "v190",   # Wealth index
    "v714",   # Currently working
    "v151",   # Sex of household head
    "v481",   # Health insurance
    "v208"    # Number of births in last 5 years
]

for var in feature_candidates:
    print("\n" + "=" * 60)
    print(f"{var} → {meta.column_names_to_labels.get(var)}")
    
    print("\nValue counts:")
    print(df[var].value_counts(dropna=False).sort_index())


v012 → respondent's current age

Value counts:
v012
15      17
16      69
17     245
18     453
19     409
20    1163
21     586
22     931
23     807
24     719
25    1868
26     911
27     995
28    1099
29     744
30    1817
31     645
32     906
33     698
34     604
35    1405
36     613
37     533
38     671
39     400
40     799
41     287
42     337
43     216
44     135
45     349
46     114
47     100
48      96
49      51
Name: count, dtype: int64

v025 → type of place of residence

Value counts:
v025
1     7710
2    14082
Name: count, dtype: int64

v106 → highest educational level

Value counts:
v106
0    9527
1    3410
2    7064
3    1791
Name: count, dtype: int64

v190 → wealth index combined

Value counts:
v190
1    5025
2    4905
3    4586
4    4025
5    3251
Name: count, dtype: int64

v714 → respondent currently working

Value counts:
v714
0     6977
1    14815
Name: count, dtype: int64

v151 → sex of household head

Value counts:
v151
1    19512
2     2280
Name: coun

In [19]:
# First Baseline feature set

baseline_features = [
    "v012",  # Age
    "v025",  # Urban/rural residence
    "v016",  # Education
    "v190",  # Wealth index
    "v714",  # Currently working
]

client_variable = "v024"

target_variable = "m15_1"

In [20]:
# Creating a temporary modelling subset

temp_df = df[df["m15_1"].notna()].copy()

print("Total eligible observations:", len(temp_df))

print("\nObservations per region:")
print(temp_df["v024"].value_counts().sort_index())

print("\nHome vs Facility delivery per region:")

print(
    pd.crosstab(
        temp_df["v024"],
        facility_delivery[temp_df.index]
    )
)

Total eligible observations: 21792

Observations per region:
v024
1    3875
2    4506
3    6309
4    2365
5    2174
6    2563
Name: count, dtype: int64

Home vs Facility delivery per region:
m15_1   0.0   1.0
v024             
1      1841  1997
2      3329  1169
3      5318   989
4       451  1887
5      1052   998
6       450  1961


In [21]:
# Checking whether code 96 explains the discrepancy

print("Records with m15_1 = 96 by region:\n")

print(
    temp_df[temp_df["m15_1"] == 96]
    .groupby("v024")
    .size()
)

print("\nTotal records coded as 96:")
print((temp_df["m15_1"] == 96).sum())

Records with m15_1 = 96 by region:

v024
1     37
2      8
3      2
4     27
5    124
6    152
dtype: int64

Total records coded as 96:
350


In [22]:
# Creating a clean binary facility-delivery target

home_codes = [11, 12]

facility_codes = [21, 22, 23, 26, 31, 36]

# Starting with the eligible population
model_df = df[df["m15_1"].notna()].copy()

# Creating the binary target
model_df["facility_delivery"] = model_df["m15_1"].map(
    lambda x: 0 if x in home_codes
    else 1 if x in facility_codes
    else None
)

# removing observations that cannot be clearly classified
model_df = model_df.dropna(subset=["facility_delivery"]).copy()

# Converting target to integer
model_df["facility_delivery"] = model_df["facility_delivery"].astype(int)

print("Final modelling observations:", len(model_df))

print("\nTarget distribution:")
print(model_df["facility_delivery"].value_counts())

print("\nTarget Percentage:")
print(
    model_df["facility_delivery"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)
      


Final modelling observations: 21442

Target distribution:
facility_delivery
0    12441
1     9001
Name: count, dtype: int64

Target Percentage:
facility_delivery
0    58.02
1    41.98
Name: proportion, dtype: float64


In [23]:
# Validating the final dataset across our  six clients

# Number of observation per federated client

print("Observations per region:")
print(model_df["v024"].value_counts().sort_index())


# Home vs Facility delivery by region

print("\nHome vs Facility delivery by region:")

regional_distribution = pd.crosstab(
    model_df["v024"],
    model_df["facility_delivery"]
)

print(regional_distribution)


# Percentage distribution of region

print("\nPercentage distribution by region:")

regional_percentage = (
    pd.crosstab(
        model_df["v024"],
        model_df["facility_delivery"],
        normalize="index"
    )
    * 100
).round(2)

print(regional_percentage)


Observations per region:
v024
1    3838
2    4498
3    6307
4    2338
5    2050
6    2411
Name: count, dtype: int64

Home vs Facility delivery by region:
facility_delivery     0     1
v024                         
1                  1841  1997
2                  3329  1169
3                  5318   989
4                   451  1887
5                  1052   998
6                   450  1961

Percentage distribution by region:
facility_delivery      0      1
v024                           
1                  47.97  52.03
2                  74.01  25.99
3                  84.32  15.68
4                  19.29  80.71
5                  51.32  48.68
6                  18.66  81.34


In [24]:
# Defining the variables needed for our first experiment

features = [
    "v012",  # Age
    "v025",  # Urban/rural residence
    "v106",  # Education level
    "v190",  # Wealth index
    "v714",  # Currently working
]

client_column = "v024"
target_column = "facility_delivery"

# Creating the clean project Dataset

project_df = model_df[
    features + [client_column, target_column]
].copy()

# Check the result

print("Project dataset shape:", project_df.shape)

print("\nCloumns:")
print(project_df.columns.tolist())

Project dataset shape: (21442, 7)

Cloumns:
['v012', 'v025', 'v106', 'v190', 'v714', 'v024', 'facility_delivery']


In [25]:
print("Data types:")
print(project_df.dtypes)

print("\n" + "=" * 50)

print("Missing values:")
print(project_df.isnull().sum())

print("\n" + "=" * 50)

print("Unique values per column:")
print(project_df.nunique())

Data types:
v012                 int64
v025                 int64
v106                 int64
v190                 int64
v714                 int64
v024                 int64
facility_delivery    int64
dtype: object

Missing values:
v012                 0
v025                 0
v106                 0
v190                 0
v714                 0
v024                 0
facility_delivery    0
dtype: int64

Unique values per column:
v012                 35
v025                  2
v106                  4
v190                  5
v714                  2
v024                  6
facility_delivery     2
dtype: int64


In [26]:
categorical_features = [
    "v025",
    "v106",
    "v190",
    "v714"
]

for feature in categorical_features:
    
    print("\n" + "=" * 70)
    print(f"{feature} distribution by region")
    print("=" * 70)
    
    distribution = pd.crosstab(
        project_df["v024"],
        project_df[feature],
        normalize="index"
    ) * 100
    
    print(distribution.round(2))


v025 distribution by region
v025      1      2
v024              
1     30.88  69.12
2     20.21  79.79
3     24.29  75.71
4     65.14  34.86
5     34.59  65.41
6     69.35  30.65

v106 distribution by region
v106      0      1      2      3
v024                            
1     35.96  19.70  34.24  10.11
2     66.72  13.61  16.45   3.22
3     74.12  11.29  11.70   2.89
4      3.34  20.27  63.13  13.26
5      6.00  17.76  63.37  12.88
6     10.12  16.84  53.21  19.83

v190 distribution by region
v190      1      2      3      4      5
v024                                   
1     16.52  24.02  24.99  20.11  14.36
2     40.57  27.68  17.99  10.47   3.29
3     35.17  30.73  18.15   9.96   5.99
4      5.35  10.82  26.69  33.32  23.82
5      3.61  14.15  26.29  28.15  27.80
6      5.02   9.00  17.30  28.91  39.78

v714 distribution by region
v714      0      1
v024              
1     29.05  70.95
2     38.24  61.76
3     46.55  53.45
4     18.82  81.18
5     21.41  78.59
6     11.07  88

In [27]:
## Completing Feature Distribution

# Age distribution summary by region

age_summary = project_df.groupby("v024")["v012"].agg(
    ["count", "mean", "median", "std", "min", "max"]
).round(2)

print(age_summary)

region_names = {
    1: "North Central",
    2: "North East",
    3: "North West",
    4: "South East",
    5: "South South",
    6: "South West"
}

age_summary_named = age_summary.copy()
age_summary_named.index = age_summary_named.index.map(region_names)

print(age_summary_named)

      count   mean  median   std  min  max
v024                                      
1      3838  29.46    29.0  6.87   15   49
2      4498  28.91    28.0  7.44   15   49
3      6307  29.17    28.0  7.56   15   49
4      2338  31.29    31.0  6.75   15   49
5      2050  30.32    30.0  6.83   15   49
6      2411  31.04    31.0  6.55   15   49
               count   mean  median   std  min  max
v024                                               
North Central   3838  29.46    29.0  6.87   15   49
North East      4498  28.91    28.0  7.44   15   49
North West      6307  29.17    28.0  7.56   15   49
South East      2338  31.29    31.0  6.75   15   49
South South     2050  30.32    30.0  6.83   15   49
South West      2411  31.04    31.0  6.55   15   49


In [28]:
# Create a dictionary containing the data for each region

region_names = {
    1: "North Central",
    2: "North East",
    3: "North West",
    4: "South East",
    5: "South South",
    6: "South West"
}

client_data = {}

for region_code, region_name in region_names.items():

    client_df = project_df[
        project_df["v024"] == region_code
    ].copy()

    client_data[region_name] = client_df

    print(
        f"{region_name}: {client_df.shape[0]} observations"
    )

North Central: 3838 observations
North East: 4498 observations
North West: 6307 observations
South East: 2338 observations
South South: 2050 observations
South West: 2411 observations


In [29]:
## Spliting Each Client into Training and Test Data

# Define the feature columns and target

features = [
    "v012",  # Age
    "v025",  # Residence
    "v106",  # Education
    "v190",  # Wealth
    "v714"   # Employment
]

target = "facility_delivery"


# Dictionary to store train/test data for each client

client_splits = {}


for region_name, client_df in client_data.items():

    # Separate features and target
    X = client_df[features].copy()
    y = client_df[target].copy()

    # Split the client's data
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    # Store the split
    client_splits[region_name] = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }

    print(f"\n{region_name}")

    print(f"Training samples: {len(X_train)}")
    print(f"Testing samples:  {len(X_test)}")

    print("\nTraining target distribution:")
    print(y_train.value_counts(normalize=True).round(4))

    print("\nTesting target distribution:")
    print(y_test.value_counts(normalize=True).round(4))


North Central
Training samples: 3070
Testing samples:  768

Training target distribution:
facility_delivery
1    0.5202
0    0.4798
Name: proportion, dtype: float64

Testing target distribution:
facility_delivery
1    0.5208
0    0.4792
Name: proportion, dtype: float64

North East
Training samples: 3598
Testing samples:  900

Training target distribution:
facility_delivery
0    0.7401
1    0.2599
Name: proportion, dtype: float64

Testing target distribution:
facility_delivery
0    0.74
1    0.26
Name: proportion, dtype: float64

North West
Training samples: 5045
Testing samples:  1262

Training target distribution:
facility_delivery
0    0.8432
1    0.1568
Name: proportion, dtype: float64

Testing target distribution:
facility_delivery
0    0.8431
1    0.1569
Name: proportion, dtype: float64

South East
Training samples: 1870
Testing samples:  468

Training target distribution:
facility_delivery
1    0.807
0    0.193
Name: proportion, dtype: float64

Testing target distribution:
facil

In [30]:
# Combine ONLY the training data from all clients
# This will be used ONLY to fit preprocessing parameters

combined_X_train = pd.concat(
    [
        client_splits[region]["X_train"]
        for region in client_splits
    ],
    axis=0
)

print("Combined training shape:", combined_X_train.shape)

print("\nFirst 5 rows:")
display(combined_X_train.head())

Combined training shape: (17151, 5)

First 5 rows:


,v012,v025,v106,v190,v714
4230,31,2,3,5,1
930,22,2,1,3,1
2534,38,1,2,4,1
6357,27,2,0,2,1
503,25,2,2,3,1


In [31]:
# Create the scaler

age_scaler = StandardScaler()

# Fit the scaler using ONLY the combined training data

age_scaler.fit(combined_X_train[["v012"]])

# Show the learned parameters

print("Training age mean:", age_scaler.mean_[0])
print("Training age standard deviation:", age_scaler.scale_[0])

Training age mean: 29.715235263249955
Training age standard deviation: 7.159848955150018


In [32]:
## Creating our preprocessing Function
"""
1. Standardize age
2. Converting residence to binary
3. Keeping the other varibales numerical
"""

def preprocess_features(X, scaler):
    
    # Create a copy so we don't modify the original data
    X_processed = X.copy()
    
    # Standardize age
    X_processed["v012"] = scaler.transform(
        X_processed[["v012"]]
    ).flatten()
    
    # Convert residence
    # 1 = Urban
    # 2 = Rural
    
    X_processed["v025"] = X_processed["v025"].map({
        1: 1,
        2: 0
    })
    
    return X_processed

In [33]:
## Tesinkg the function on one client

# Getting Northing Central data

X_nc_train = client_splits["North Central"]["X_train"]

# Preprocess it
X_nc_train_processed = preprocess_features(
    X_nc_train,
    age_scaler
)

# Display the first 5 rows
display(X_nc_train_processed.head())

,v012,v025,v106,v190,v714
4230,0.179440,0,3,5,1
930,-1.077570,0,1,3,1
2534,1.157114,1,2,4,1
6357,-0.379231,0,0,2,1
503,-0.658566,0,2,3,1


In [34]:
print("Mean of scaled age:")
print(X_nc_train_processed["v012"].mean())

print("\nStandard deviation of scaled age:")
print(X_nc_train_processed["v012"].std())

Mean of scaled age:
-0.0444831440209495

Standard deviation of scaled age:
0.9509185548324696


In [35]:
## Process all six Regional clients

for region in client_splits:
    
    # Process training features
    client_splits[region]["X_train_processed"] = preprocess_features(
        client_splits[region]["X_train"],
        age_scaler
    )
    
    # Process testing features
    client_splits[region]["X_test_processed"] = preprocess_features(
        client_splits[region]["X_test"],
        age_scaler
    )

print("Preprocessing completed for all six regional clients.")

# verification

for region in client_splits:
    
    X_train = client_splits[region]["X_train_processed"]
    X_test = client_splits[region]["X_test_processed"]
    
    print(f"\n{region}")
    print(f"Training shape: {X_train.shape}")
    print(f"Testing shape: {X_test.shape}")
    print(f"Missing values: {X_train.isnull().sum().sum()}")


Preprocessing completed for all six regional clients.

North Central
Training shape: (3070, 5)
Testing shape: (768, 5)
Missing values: 0

North East
Training shape: (3598, 5)
Testing shape: (900, 5)
Missing values: 0

North West
Training shape: (5045, 5)
Testing shape: (1262, 5)
Missing values: 0

South East
Training shape: (1870, 5)
Testing shape: (468, 5)
Missing values: 0

South South
Training shape: (1640, 5)
Testing shape: (410, 5)
Missing values: 0

South West
Training shape: (1928, 5)
Testing shape: (483, 5)
Missing values: 0


In [36]:

# Combine processed training data from all regions
X_central_train = pd.concat(
    [
        client_splits[region]["X_train_processed"]
        for region in client_splits
    ],
    axis=0
)

# Combine training targets
y_central_train = pd.concat(
    [
        client_splits[region]["y_train"]
        for region in client_splits
    ],
    axis=0
)

print("Centralized training shape:", X_central_train.shape)
print("Centralized target shape:", y_central_train.shape)

print("\nTarget distribution:")
print(y_central_train.value_counts(normalize=True))

Centralized training shape: (17151, 5)
Centralized target shape: (17151,)

Target distribution:
facility_delivery
0    0.580316
1    0.419684
Name: proportion, dtype: float64


In [37]:
## Builing the Neural Network

# Creating our neural network

class FacilityDeliveryModel(nn.Module):

    def __init__(self):
        super(FacilityDeliveryModel, self).__init__()

        self.network = nn.Sequential(

            # Input layer: 5 features
            nn.Linear(5, 16),
            nn.ReLU(),

            # Hidden layer
            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)
    
# Creating the Model

model = FacilityDeliveryModel()

print(model)


FacilityDeliveryModel(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)


In [38]:
# Building the local trnaining and evaluation functions

# Feature columns used by the model
feature_columns = [
    "v012",
    "v025",
    "v106",
    "v190",
    "v714"
]

# Converting centralized training data to Pytorch tensors

X_train_tensor = torch.tensor(
    X_central_train[feature_columns].values,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_central_train.values,
    dtype=torch.float32
).view(-1, 1)

print("Training features shape:", X_train_tensor.shape)
print("Training labels shape:", y_train_tensor.shape)

Training features shape: torch.Size([17151, 5])
Training labels shape: torch.Size([17151, 1])


In [39]:
# Defining the loss function and Optimizer

criterion = nn.BCEWithLogitsLoss()

# Defining the optimizer

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [40]:
outputs = model(X_train_tensor)

print("Output shape:", outputs.shape)
print("\nFirst 5 raw outputs:")
print(outputs[:5])

Output shape: torch.Size([17151, 1])

First 5 raw outputs:
tensor([[0.5525],
        [0.4614],
        [0.4323],
        [0.3923],
        [0.4862]], grad_fn=<SliceBackward0>)


In [41]:
# calculate the loss

loss = criterion(outputs, y_train_tensor)

print("Initial loss:", loss.item())

Initial loss: 0.7333701848983765


In [42]:
## Backpropagation: where the learning actually happens

# Clear any previous gradients
optimizer.zero_grad()

# Calculate gradients
loss.backward()

# Update the model's weights
optimizer.step()

print("One training step completed successfully!")

One training step completed successfully!


In [43]:
## Verifying if the model improved

# Making predictions again after the weight update

outputs_after = model(X_train_tensor)

# Calculating the new loss
loss_after = criterion(outputs_after, y_train_tensor)

print("Loss before update:", loss.item())
print("Loss after update:", loss_after.item() )


Loss before update: 0.7333701848983765
Loss after update: 0.7324221730232239


The loss decreased because backpropagation calculated how the model's weights contributed to the error, and Adam used those gradients to adjust the weights toward a lower loss.


I am stopping here im centralised training because it is only to verify that I understand and have a functioning PyTorch model.

In [44]:
## Building the local training loop

# Creating a PyTorch Dataset

class FacilityDeliveryDataset(Dataset): # Dataset is PyTorch way of representing training data
    def __init__(self, X, y):
        self.X = torch.tensor(
            X.values,
            dtype=torch.float32
        ) 

        self.y = torch.tensor(
            y.values,
            dtype=torch.float32
        ).view(-1, 1)  # converts the facility-delivery target into the shape our model expects

    def __len__(self):
        return len(self.X)  # counts the training examples

    def __getitem__(self, index):
        return self.X[index], self.y[index]  # gives the feature and target for each example index

In [45]:
# Testing one Client (North Central) first

nc_dataset = FacilityDeliveryDataset(
    client_splits["North Central"]["X_train_processed"],
    client_splits["North Central"]["y_train"]
)

print("Number od samples:", len(nc_dataset))
print("First sample")
print(nc_dataset[0])

Number od samples: 3070
First sample
(tensor([0.1794, 0.0000, 3.0000, 5.0000, 1.0000]), tensor([1.]))


In [46]:
# creating the dataloader

nc_loader = DataLoader(
    nc_dataset,
    batch_size=32,
    shuffle=True
)

print("Number of batches:", len(nc_loader))

Number of batches: 96


In [47]:
# Writing the local training function

def train_local(model, X_train, y_train, epochs=1, batch_size=32):

    dataset = FacilityDeliveryDataset(
        X_train,
        y_train
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    model.train()

    for epoch in range(epochs):

        total_loss = 0.0

        for X_batch, y_batch in loader:

            optimizer.zero_grad()  # clear gradients from previous batch

            outputs = model(X_batch)  # forward pass

            loss = criterion(
                outputs,
                y_batch
            )  # shows how wrong the predictions are

            loss.backward()  # perfoms backpropagation and calculates the gradients

            optimizer.step()  # using the gradients to updates the model's weights

            total_loss += loss.item()

        average_loss = total_loss / len(loader)

        print(
                f"Epoch {epoch + 1}/{epochs}, "
                f"Loss: {average_loss:.4f}"
        )

In [48]:
nc_model = FacilityDeliveryModel()

print("Model created successfully.")

Model created successfully.


In [49]:
print("Starting local training...")

train_local(
    nc_model,
    client_splits["North Central"]["X_train_processed"],
    client_splits["North Central"]["y_train"],
    epochs=1
)

print("Training finished.")

Starting local training...
Epoch 1/1, Loss: 0.6821
Training finished.


## FedAvg prototyping section

In [50]:
## Understanding FedAvg manually first before applying Flower

# Creating a common starting model

global_model = FacilityDeliveryModel()

print(
    "Initial weight:", global_model.network[0].weight[0][0].item()
)

# Giving each client a copy

nc_model = copy.deepcopy(global_model)
ne_model = copy.deepcopy(global_model)

# Training each client locally

train_local(
    nc_model,
    client_splits["North Central"]["X_train_processed"],
    client_splits["North Central"]["y_train"],
    epochs=1
    )

train_local(
    ne_model,
    client_splits["North East"]["X_train_processed"],
    client_splits["North East"]["y_train"],
    epochs=1
    )

# Comparing the trained weights

print("Global model weights:", global_model.network[0].weight[0][0].item())
print("North Central weights:", nc_model.network[0].weight[0][0].item())
print("North East weights:", ne_model.network[0].weight[0][0].item())

Initial weight: -0.22581829130649567
Epoch 1/1, Loss: 0.6738
Epoch 1/1, Loss: 0.6120
Global model weights: -0.22581829130649567
North Central weights: -0.2460593432188034
North East weights: -0.29265937209129333


In [51]:
# Manually calculating FedAvg

nc_weight = nc_model.network[0].weight[0][0].item()
ne_weight = ne_model.network[0].weight[0][0].item()

nc_samples = 3070
ne_samples = 3598

fedavg_weight = (
    (nc_samples / (nc_samples + ne_samples)) * nc_weight
    +
    (ne_samples / (nc_samples + ne_samples)) * ne_weight
)

print("North Central contribution:",
      nc_samples / (nc_samples + ne_samples))

print("North East contribution:",
      ne_samples / (nc_samples + ne_samples))

print("FedAvg weight:",
      fedavg_weight)

North Central contribution: 0.46040791841631673
North East contribution: 0.5395920815836832
FedAvg weight: -0.27120434979997


In [52]:
## Buidling for six-client FedAvg function

# Creating the FedAvg function

def fedavg(models, sample_counts):

    total_samples = sum(sample_counts)

    global_state = copy.deepcopy(
        models[0].state_dict()
    )

    for key in global_state:

        global_state[key] = sum(
            (sample_counts[i] / total_samples)
            * models[i].state_dict()[key]
            for i in range(len(models))
        )

    return global_state

In [53]:
# Testing it with the two models we already trained

fedavg_state = fedavg(
    [nc_model, ne_model],
    [3070, 3598]
)

print(
    "FedAvg weight:",
    fedavg_state["network.0.weight"][0][0].item()
)

FedAvg weight: -0.2712043523788452


## Actual Federated Learning Experiment

The following section begins the actual six-client federated learning
experiment using Nigeria's six geopolitical zones.

Each client starts from the same global model and trains only on its
local regional training data. The server aggregates client model
parameters using FedAvg.

In [54]:
# Fresh global model for the actual experiment

global_experiment = FacilityDeliveryModel()

print(global_model)

FacilityDeliveryModel(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)


In [55]:
# Creating six identical client models

client_models = {}

for region in region_names.values():
    client_models[region] = copy.deepcopy(global_model)

print("Number of Clients:", len(client_models))

# Verifying that they really are identical

for region, client_model in client_models.items():

    weight = client_model.network[0].weight[0][0].item()

    print(
        region,
        "-",
        weight
    )


Number of Clients: 6
North Central - -0.22581829130649567
North East - -0.22581829130649567
North West - -0.22581829130649567
South East - -0.22581829130649567
South South - -0.22581829130649567
South West - -0.22581829130649567


In [56]:
# Training each of the six clients locally

for region_code, region in region_names.items():

    print(f"\nTraining {region}...")

    train_local(
        client_models[region],
        client_splits[region]["X_train_processed"],
        client_splits[region]["y_train"],
        epochs=1,
        batch_size=32

    )


Training North Central...
Epoch 1/1, Loss: 0.6740

Training North East...
Epoch 1/1, Loss: 0.6127

Training North West...
Epoch 1/1, Loss: 0.5381

Training South East...
Epoch 1/1, Loss: 0.6800

Training South South...
Epoch 1/1, Loss: 0.6857

Training South West...
Epoch 1/1, Loss: 0.6777


In [57]:
# actual fedAvg aggregation

sample_counts = [
    3070,
    3598,
    5045,
    1870,
    1640,
    1928
]

models = [
    client_models["North Central"],
    client_models["North East"],
    client_models["North West"],
    client_models["South East"],
    client_models["South South"],
    client_models["South West"]
]

new_global_state = fedavg(
    models,
    sample_counts
)

print(
    "Aggregated weight:",
    new_global_state["network.0.weight"][0][0].item()
)

Aggregated weight: -0.2679392099380493


In [58]:
# Updating the gloval model

global_model.load_state_dict(new_global_state)

print("Updated global model weight:", global_model.network[0].weight[0][0].item())

Updated global model weight: -0.2679392099380493


In [59]:
def evaluate_model(model, X_test, y_test):

    dataset = FacilityDeliveryDataset(
        X_test,
        y_test
    )

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False
    )

    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            outputs = model(X_batch)

            probabilities = torch.sigmoid(outputs)

            predictions = (
                probabilities >= 0.5
            ).float()

            all_predictions.extend(
                predictions.cpu().numpy().flatten()
            )

            all_targets.extend(
                y_batch.cpu().numpy().flatten()
            )

    accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    precision = precision_score(
        all_targets,
        all_predictions,
        zero_division=0
    )

    recall = recall_score(
        all_targets,
        all_predictions,
        zero_division=0
    )

    f1 = f1_score(
        all_targets,
        all_predictions,
        zero_division=0
    )

    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "macro_f1": macro_f1
    }

In [60]:
# Testing global model on North Central 

nc_metrics = evaluate_model(
    global_model,
    client_splits["North Central"]["X_test_processed"],
    client_splits["North Central"]["y_test"]
)

print(nc_metrics)

{'accuracy': 0.4791666666666667, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'macro_f1': 0.323943661971831}


In [61]:
print("Predicted class distribution")

# Recreating the test dataset
nc_dataset = FacilityDeliveryDataset(
    client_splits["North Central"]['X_test_processed'],
    client_splits["North Central"]["y_test"]
)

nc_loader = DataLoader(
    nc_dataset,
    batch_size=32,
    shuffle=False
)

global_model.eval()

all_predictions = []

with torch.no_grad():

    for X_batch, y_batch in nc_loader:

        outputs = global_model(X_batch)

        probabilities = torch.sigmoid(outputs)

        predictions = (
            probabilities >= 0.5
        ).float()

        all_predictions.extend(
            predictions.cpu().numpy().flatten()
        )

print(
    "Class 0:",
    all_predictions.count(0.0)
)

print(
    "Class 1:",
    all_predictions.count(1.0)
)

Predicted class distribution
Class 0: 768
Class 1: 0


In [62]:
# Evaluating all six regions

round1_results = {}

for region in region_names.values():

    metrics = evaluate_model(
        global_model,
        client_splits[region]["X_test_processed"],
        client_splits[region]["y_test"]

    )

    round1_results[region] = metrics

    print(f"\n{region}")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1: {metrics['f1']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")


North Central
Accuracy: 0.4792
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
Macro F1: 0.3239

North East
Accuracy: 0.7400
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
Macro F1: 0.4253

North West
Accuracy: 0.8431
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
Macro F1: 0.4574

South East
Accuracy: 0.1923
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
Macro F1: 0.1613

South South
Accuracy: 0.5122
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
Macro F1: 0.3387

South West
Accuracy: 0.1863
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
Macro F1: 0.1571


### Round 1 observation

After one local training epoch and one FedAvg aggregation, the global
model showed substantial variation in performance across geopolitical
zones. Accuracy ranged from 18.84% in South West to 84.31% in North West,
while positive-class F1 was near zero across all regions.

This early-round result should not be interpreted as final model
performance. The strong regional variation is consistent with the
substantial differences in regional target distributions and motivates
evaluation across multiple federated rounds using metrics beyond
accuracy.

## Multi-Round FedAvg Experiment

In [63]:
def run_fedavg_experiment(
    initial_model,
    client_splits,
    region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    seed=42
):
    set_seed(seed)

    global_model = copy.deepcopy(initial_model)

    results = []

    for round_number in range(1, num_rounds + 1):

        print(f"\n{'=' * 50}")
        print(f"Federated Round {round_number}")
        print(f"{'=' * 50}")

        # Create a dictionary for the six client models
        client_models = {}

        sample_counts = []

        # Give every client a fresh copy of the current global model
        for region_code, region in region_names.items():

            client_models[region] = copy.deepcopy(
                global_model
            )

        # Train each client locally
        for region_code, region in region_names.items():

            print(f"\nTraining {region}...")

            train_local(
                client_models[region],
                client_splits[region]["X_train_processed"],
                client_splits[region]["y_train"],
                epochs=local_epochs,
                batch_size=batch_size
            )

            sample_counts.append(
                len(client_splits[region]["y_train"])
            )

        # Collect the six trained models
        models = [
            client_models[region]
            for region in region_names.values()
        ]

        # Perform FedAvg
        new_global_state = fedavg(
            models,
            sample_counts
        )

        # Update the global model
        global_model.load_state_dict(
            new_global_state
        )

        # Evaluate the new global model
        for region_code, region in region_names.items():

            metrics = evaluate_model(
                global_model,
                client_splits[region]["X_test_processed"],
                client_splits[region]["y_test"]
            )

            results.append({
                "round": round_number,
                "region": region,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "macro_f1": metrics["macro_f1"]
            })

            print(
                f"{region} | "
                f"Accuracy: {metrics['accuracy']:.4f} | "
                f"Macro F1: {metrics['macro_f1']:.4f}"
            )

    return global_model, results

In [64]:
#Creating a completely fresh model

experiment_initial_model = FacilityDeliveryModel()

print(
    "Experiment starting model weight:", 
    experiment_initial_model.network[0].weight[0][0].item()
)

Experiment starting model weight: 0.44427406787872314


In [65]:
fedavg_global_model, fedavg_results = run_fedavg_experiment(
    initial_model=experiment_initial_model,
    client_splits=client_splits,
    region_names=region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32
)


Federated Round 1

Training North Central...
Epoch 1/1, Loss: 0.7173

Training North East...
Epoch 1/1, Loss: 0.6157

Training North West...
Epoch 1/1, Loss: 0.5163

Training South East...
Epoch 1/1, Loss: 0.8199

Training South South...
Epoch 1/1, Loss: 0.7295

Training South West...
Epoch 1/1, Loss: 0.8236
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.7400 | Macro F1: 0.4253
North West | Accuracy: 0.8431 | Macro F1: 0.4574
South East | Accuracy: 0.1923 | Macro F1: 0.1613
South South | Accuracy: 0.5122 | Macro F1: 0.3387
South West | Accuracy: 0.1863 | Macro F1: 0.1571

Federated Round 2

Training North Central...
Epoch 1/1, Loss: 0.7046

Training North East...
Epoch 1/1, Loss: 0.5852

Training North West...
Epoch 1/1, Loss: 0.4799

Training South East...
Epoch 1/1, Loss: 0.7927

Training South South...
Epoch 1/1, Loss: 0.7128

Training South West...
Epoch 1/1, Loss: 0.7960
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.740

In [66]:
# Saving the result as a dataframe

fedavg_results_df = pd.DataFrame(
    fedavg_results
)

fedavg_results_df.head()

,round,region,accuracy,precision,recall,f1,macro_f1
0,1,North Central,0.479167,0.0,0.0,0.0,0.323944
1,1,North East,0.740000,0.0,0.0,0.0,0.425287
2,1,North West,0.843106,0.0,0.0,0.0,0.457438
3,1,South East,0.192308,0.0,0.0,0.0,0.161290
4,1,South South,0.512195,0.0,0.0,0.0,0.338710


In [67]:
print(
    fedavg_results_df.shape
)

print(
    fedavg_results_df["round"].unique()
)

(30, 7)
[1 2 3 4 5]


In [68]:
# Extracting important FedAvg summary

fedavg_summary = (
    fedavg_results_df
    .groupby("round")
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_macro_f1=("macro_f1", "mean"),
        min_macro_f1=("macro_f1", "min"),
        max_macro_f1=("macro_f1", "max")
    )
    .reset_index()
)

fedavg_summary


fedavg_final = fedavg_results_df[
    fedavg_results_df["round"] == 5
]

fedavg_final

,round,region,accuracy,precision,recall,f1,macro_f1
24,5,North Central,0.479167,0.0,0.0,0.0,0.323944
25,5,North East,0.740000,0.0,0.0,0.0,0.425287
26,5,North West,0.843106,0.0,0.0,0.0,0.457438
27,5,South East,0.192308,0.0,0.0,0.0,0.161290
28,5,South South,0.512195,0.0,0.0,0.0,0.338710
29,5,South West,0.186335,0.0,0.0,0.0,0.157068


## FedProx training function

In [69]:
def train_local_fedprox(
    model,
    global_model,
    X_train,
    y_train,
    mu=0.01,
    epochs=1,
    batch_size=32
):
    dataset = FacilityDeliveryDataset(
        X_train,
        y_train
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    model.train()
    global_model.eval()

    # Keep a fixed copy of the global model parameters
    global_params = [
        param.detach().clone()
        for param in global_model.parameters()
    ]

    for epoch in range(epochs):

        total_loss = 0.0

        for X_batch, y_batch in loader:

            optimizer.zero_grad()

            # Classification loss
            outputs = model(X_batch)

            classification_loss = criterion(
                outputs,
                y_batch
            )

            # Proximal penalty
            proximal_loss = 0.0

            for local_param, global_param in zip(
                model.parameters(),
                global_params
            ):
                proximal_loss += torch.sum(
                    (local_param - global_param) ** 2
                )

            proximal_loss = (mu / 2) * proximal_loss

            # FedProx objective
            loss = (
                classification_loss
                + proximal_loss
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        average_loss = (
            total_loss / len(loader)
        )

        print(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Loss: {average_loss:.4f}"
        )

In [70]:
# Testing one client

# creating a fresh global model
test_global_model = FacilityDeliveryModel()

# Creating a client model starting from the same global model
test_client_model = copy.deepcopy(test_global_model)

#Training the client using FedProx
train_local_fedprox(
    test_client_model,
    test_global_model,
    client_splits["North Central"]["X_train_processed"],
    client_splits["North Central"]["y_train"],
    mu=0.01,
    epochs=1,
    batch_size=32
)

Epoch 1/1, Loss: 0.6822


In [71]:
# Check the ordinary classification loss on the freshly initialized client
test_global_model = FacilityDeliveryModel()
test_client_model = copy.deepcopy(test_global_model)

dataset = FacilityDeliveryDataset(
    client_splits["North Central"]["X_train_processed"],
    client_splits["North Central"]["y_train"]
)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

criterion = nn.BCEWithLogitsLoss()

test_client_model.eval()

X_batch, y_batch = next(iter(loader))

with torch.no_grad():
    outputs = test_client_model(X_batch)
    classification_loss = criterion(outputs, y_batch)

print("Classification loss:", classification_loss.item())

Classification loss: 0.6957448124885559


In [72]:
# a diagnostic version for one client

test_global_model = FacilityDeliveryModel()
test_client_model = copy.deepcopy(test_global_model)

dataset = FacilityDeliveryDataset(
    client_splits["North Central"]["X_train_processed"],
    client_splits["North Central"]["y_train"]
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    test_client_model.parameters(),
    lr=0.001
)

test_client_model.train()
test_global_model.eval()

global_params = [
    param.detach().clone()
    for param in test_global_model.parameters()
]

X_batch, y_batch = next(iter(loader))

optimizer.zero_grad()

outputs = test_client_model(X_batch)

classification_loss = criterion(outputs, y_batch)

proximal_loss = 0.0

for local_param, global_param in zip(
    test_client_model.parameters(),
    global_params
):
    proximal_loss += torch.sum(
        (local_param - global_param) ** 2
    )

mu = 0.01

proximal_loss = (mu / 2) * proximal_loss

total_loss = classification_loss + proximal_loss

print("Classification loss:", classification_loss.item())
print("Proximal loss:", proximal_loss.item())
print("Total FedProx loss:", total_loss.item())

Classification loss: 0.6772440671920776
Proximal loss: 0.0
Total FedProx loss: 0.6772440671920776


In [73]:
# Fresh models
test_global_model = FacilityDeliveryModel()
test_client_model = copy.deepcopy(test_global_model)

dataset = FacilityDeliveryDataset(
    client_splits["North Central"]["X_train_processed"],
    client_splits["North Central"]["y_train"]
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    test_client_model.parameters(),
    lr=0.001
)

test_client_model.train()
test_global_model.eval()

# Save the global parameters
global_params = [
    param.detach().clone()
    for param in test_global_model.parameters()
]

mu = 0.01

# One batch
X_batch, y_batch = next(iter(loader))

optimizer.zero_grad()

outputs = test_client_model(X_batch)

classification_loss = criterion(
    outputs,
    y_batch
)

proximal_loss = 0.0

for local_param, global_param in zip(
    test_client_model.parameters(),
    global_params
):
    proximal_loss += torch.sum(
        (local_param - global_param) ** 2
    )

proximal_loss = (mu / 2) * proximal_loss

total_loss = classification_loss + proximal_loss

total_loss.backward()

# Update local model
optimizer.step()

# Measure how far the local model moved
model_distance = 0.0

for local_param, global_param in zip(
    test_client_model.parameters(),
    global_params
):
    model_distance += torch.sum(
        (local_param.detach() - global_param) ** 2
    )

print("Classification loss:", classification_loss.item())
print("Proximal loss BEFORE update:", proximal_loss.item())
print("Total loss:", total_loss.item())
print("Model distance AFTER update:", model_distance.item())

Classification loss: 0.7000107169151306
Proximal loss BEFORE update: 0.0
Total loss: 0.7000107169151306
Model distance AFTER update: 0.0001669911725912243


In [74]:
## Creating the reproducible initial model

# making the reproducible starting model
torch.manual_seed(42)  # sets random seed for model initialization

fed_experiment_initial_model = FacilityDeliveryModel() # Creates a fresh 5 -> 16 -> 8 -> 1 model

# Saving the exact initial parameters
fed_experiment_initial_state = copy.deepcopy(
    fed_experiment_initial_model.state_dict()  # independent copy of its parameters
)

print(
    "Initial model created with", len(fed_experiment_initial_state), "parameter tensors."
)

Initial model created with 6 parameter tensors.


In [75]:
# Creating the FedProx experiment function

def run_fedprox_experiment(
    initial_state,
    client_splits,
    region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    mu=0.01,
    seed=42
):
    set_seed(seed)
    
    # Create fresh global model
    global_model = FacilityDeliveryModel()

    # Start from the controlled initial weights
    global_model.load_state_dict(
        copy.deepcopy(initial_state)
    )

    results = []

    for round_number in range(1, num_rounds + 1):

        print("\n" + "=" * 50)
        print(f"FedProx Round {round_number}")
        print("=" * 50)

        # Dictionary to store each client's local model
        client_models = {}

        # Store number of training samples for each client
        sample_counts = []

        # Give every client a copy of the current global model
        for region_code, region in region_names.items():
            client_models[region] = copy.deepcopy(
                global_model
            )

        # Local training
        for region_code, region in region_names.items():

            print(f"\nTraining {region}...")

            train_local_fedprox(
                client_models[region],
                global_model,
                client_splits[region]["X_train_processed"],
                client_splits[region]["y_train"],
                mu=mu,
                epochs=local_epochs,
                batch_size=batch_size
            )

            sample_counts.append(
                len(
                    client_splits[region]["y_train"]
                )
            )

        # Collect models in the same order as region_names
        models = [
            client_models[region]
            for region in region_names.values()
        ]

        # Aggregate client models
        new_global_state = fedavg(
            models,
            sample_counts
        )

        # Update global model
        global_model.load_state_dict(
            new_global_state
        )

        # Evaluate global model on each regional test set
        for region_code, region in region_names.items():

            metrics = evaluate_model(
                global_model,
                client_splits[region]["X_test_processed"],
                client_splits[region]["y_test"]
            )

            results.append({
                "round": round_number,
                "region": region,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "macro_f1": metrics["macro_f1"]
            })

            print(
                f"{region} | "
                f"Accuracy: {metrics['accuracy']:.4f} | "
                f"Macro F1: {metrics['macro_f1']:.4f}"
            )

    return global_model, results

In [76]:
controlled_fedavg_initial_model = FacilityDeliveryModel()

controlled_fedavg_initial_model.load_state_dict(
    copy.deepcopy(fed_experiment_initial_state)
)

<All keys matched successfully>

In [77]:
fedavg_controlled_model, fedavg_controlled_results = run_fedavg_experiment(
    initial_model=controlled_fedavg_initial_model,
    client_splits=client_splits,
    region_names=region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32
)


Federated Round 1

Training North Central...
Epoch 1/1, Loss: 0.6916

Training North East...
Epoch 1/1, Loss: 0.6722

Training North West...
Epoch 1/1, Loss: 0.5909

Training South East...
Epoch 1/1, Loss: 0.6185

Training South South...
Epoch 1/1, Loss: 0.6986

Training South West...
Epoch 1/1, Loss: 0.6189
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.7400 | Macro F1: 0.4253
North West | Accuracy: 0.8431 | Macro F1: 0.4574
South East | Accuracy: 0.1923 | Macro F1: 0.1613
South South | Accuracy: 0.5122 | Macro F1: 0.3387
South West | Accuracy: 0.1863 | Macro F1: 0.1571

Federated Round 2

Training North Central...
Epoch 1/1, Loss: 0.6824

Training North East...
Epoch 1/1, Loss: 0.6173

Training North West...
Epoch 1/1, Loss: 0.5238

Training South East...
Epoch 1/1, Loss: 0.6553

Training South South...
Epoch 1/1, Loss: 0.6939

Training South West...
Epoch 1/1, Loss: 0.6549
North Central | Accuracy: 0.4779 | Macro F1: 0.3233
North East | Accuracy: 0.740

In [78]:
fedavg_controlled_results_df = pd.DataFrame(
    fedavg_controlled_results
)

print(fedavg_controlled_results_df.shape)

(30, 7)


In [79]:
fedprox_model, fedprox_results = run_fedprox_experiment(
    initial_state=fed_experiment_initial_state,
    client_splits=client_splits,
    region_names=region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    mu=0.01
)


FedProx Round 1

Training North Central...
Epoch 1/1, Loss: 0.6915

Training North East...
Epoch 1/1, Loss: 0.6746

Training North West...
Epoch 1/1, Loss: 0.5932

Training South East...
Epoch 1/1, Loss: 0.6266

Training South South...
Epoch 1/1, Loss: 0.6982

Training South West...
Epoch 1/1, Loss: 0.6146
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.7400 | Macro F1: 0.4253
North West | Accuracy: 0.8431 | Macro F1: 0.4574
South East | Accuracy: 0.1923 | Macro F1: 0.1613
South South | Accuracy: 0.5122 | Macro F1: 0.3387
South West | Accuracy: 0.1863 | Macro F1: 0.1571

FedProx Round 2

Training North Central...
Epoch 1/1, Loss: 0.6857

Training North East...
Epoch 1/1, Loss: 0.6191

Training North West...
Epoch 1/1, Loss: 0.5276

Training South East...
Epoch 1/1, Loss: 0.6642

Training South South...
Epoch 1/1, Loss: 0.6920

Training South West...
Epoch 1/1, Loss: 0.6566
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.7400 | 

In [80]:
# 5-round FedProx experiment

fedprox_model, fedprox_results = run_fedprox_experiment(
    initial_state=fed_experiment_initial_state,
    client_splits=client_splits,
    region_names=region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    mu=0.01
)


FedProx Round 1

Training North Central...
Epoch 1/1, Loss: 0.6915

Training North East...
Epoch 1/1, Loss: 0.6746

Training North West...
Epoch 1/1, Loss: 0.5932

Training South East...
Epoch 1/1, Loss: 0.6266

Training South South...
Epoch 1/1, Loss: 0.6982

Training South West...
Epoch 1/1, Loss: 0.6146
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.7400 | Macro F1: 0.4253
North West | Accuracy: 0.8431 | Macro F1: 0.4574
South East | Accuracy: 0.1923 | Macro F1: 0.1613
South South | Accuracy: 0.5122 | Macro F1: 0.3387
South West | Accuracy: 0.1863 | Macro F1: 0.1571

FedProx Round 2

Training North Central...
Epoch 1/1, Loss: 0.6857

Training North East...
Epoch 1/1, Loss: 0.6191

Training North West...
Epoch 1/1, Loss: 0.5276

Training South East...
Epoch 1/1, Loss: 0.6642

Training South South...
Epoch 1/1, Loss: 0.6920

Training South West...
Epoch 1/1, Loss: 0.6566
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.7400 | 

In [81]:
# Creating the FedProx results DataFrame

fedprox_results_df = pd.DataFrame(fedprox_results)

print(fedprox_results_df.shape)
print(fedprox_results_df["round"].unique())

(30, 7)
[1 2 3 4 5]


In [82]:
# Creating a summary for both algorithms

fedavg_summary = (
    fedavg_controlled_results_df
    .groupby("round")
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_macro_f1=("macro_f1", "mean"),
        min_macro_f1 = ("macro_f1", "min"),
        max_macro_f1=("macro_f1", "max")
    )
    .reset_index()
)

fedprox_summary = (
    fedprox_results_df
    .groupby("round")
    .agg(
        mean_accuracy=("accuracy", "mean"),
        mean_macro_f1=("macro_f1", "mean"),
        min_macro_f1 = ("macro_f1", "min"),
        max_macro_f1=("macro_f1", "max")
    )
    .reset_index()
)

print("FedAvg")
display(fedavg_summary)

print("FedProx")
display(fedprox_summary)

FedAvg


,round,mean_accuracy,mean_macro_f1,min_macro_f1,max_macro_f1
0,1,0.492185,0.310623,0.157068,0.457438
1,2,0.492313,0.310992,0.159881,0.457438
2,3,0.691953,0.611240,0.547175,0.693547
3,4,0.740355,0.653430,0.589856,0.718082
4,5,0.740473,0.648113,0.545565,0.710370


FedProx


,round,mean_accuracy,mean_macro_f1,min_macro_f1,max_macro_f1
0,1,0.492185,0.310623,0.157068,0.457438
1,2,0.491840,0.310377,0.155594,0.457438
2,3,0.694849,0.613703,0.550011,0.695139
3,4,0.730110,0.645373,0.561014,0.723234
4,5,0.733164,0.649725,0.569103,0.719023


In [83]:
# Creating  a Round 5 comparison table

fedavg_final = (
    fedavg_controlled_results_df[
        fedavg_controlled_results_df["round"] == 5
    ]
    [["region", "accuracy", "precision", "recall", "f1", "macro_f1"]]
    .copy()
)

fedprox_final = (
    fedprox_results_df[
        fedprox_results_df["round"] == 5
    ]
    [["region", "accuracy", "precision", "recall", "f1", "macro_f1"]]
    .copy()
)

comparison = fedavg_final.merge(
    fedprox_final,
    on="region",
    suffixes=("_fedavg", "_fedprox")
)

comparison["macro_f1_difference"] = (
    comparison["macro_f1_fedprox"]
    - comparison["macro_f1_fedavg"]
)

comparison

,region,accuracy_fedavg,precision_fedavg,recall_fedavg,f1_fedavg,macro_f1_fedavg,accuracy_fedprox,precision_fedprox,recall_fedprox,f1_fedprox,macro_f1_fedprox,macro_f1_difference
0,North Central,0.671875,0.705556,0.635000,0.668421,0.671839,0.664062,0.719136,0.582500,0.643646,0.662956,-0.008883
1,North East,0.774444,0.590643,0.431624,0.498765,0.676623,0.776667,0.613793,0.380342,0.469657,0.664104,-0.012519
2,North West,0.847068,0.512690,0.510101,0.511392,0.710370,0.856577,0.546961,0.500000,0.522427,0.719023,0.008653
3,South East,0.818376,0.872774,0.907407,0.889754,0.687301,0.797009,0.898592,0.843915,0.870396,0.701208,0.013907
4,South South,0.587805,0.546269,0.915000,0.684112,0.545565,0.604878,0.562092,0.860000,0.679842,0.581959,0.036394
5,South West,0.743271,0.853018,0.826972,0.839793,0.596980,0.699793,0.848315,0.768448,0.806409,0.569103,-0.027877


In [84]:
import random

# Creating a small seed function

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.maual_seed_all(seed)

In [85]:
def create_initial_model(seed=42):
    set_seed(seed)

    model = FacilityDeliveryModel()

    return model

In [86]:
initial_model_seed42 = create_initial_model(42)

print(initial_model_seed42)

FacilityDeliveryModel(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
  )
)


In [87]:
# Freezing its exact weights

initial_state_seed42 = copy.deepcopy(
    initial_model_seed42.state_dict()
)

print(f"Saved {len(initial_state_seed42)} parameter tensors.")

Saved 6 parameter tensors.


In [88]:
# Building a fresh model with the frozen initial_state_seed42

def create_model_from_state(initial_state):
    model = FacilityDeliveryModel()

    model.load_state_dict(
        copy.deepcopy(initial_state)
    )

    return model

fedavg_start_model = create_model_from_state(
    initial_state_seed42
)

fedprox_start_model = create_model_from_state(
    initial_state_seed42
)

same_initial_weights = all(
    torch.equal(
        fedavg_start_model.state_dict()[key],
        fedprox_start_model.state_dict()[key]
    )
    for key in fedavg_start_model.state_dict()
)

print("Same initial weights:", same_initial_weights)

Same initial weights: True


In [89]:
def run_reproducible_fedavg(
    initial_state,
    client_splits,
    region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    seed=42
):
    # Reset all random generators for reproducibility
    set_seed(seed)

    # Create the global model from the exact saved initial state
    global_model = FacilityDeliveryModel()
    global_model.load_state_dict(copy.deepcopy(initial_state))

    results = []

    for round_number in range(1, num_rounds + 1):

        print("\n" + "=" * 50)
        print(f"Reproducible FedAvg Round {round_number}")
        print("=" * 50)

        client_models = {}
        sample_counts = []

        # Create a copy of the current global model for each client
        for region_code, region in region_names.items():
            client_models[region] = copy.deepcopy(global_model)

        # Local training
        for region_code, region in region_names.items():

            print(f"\nTraining {region}...")

            train_local(
                client_models[region],
                client_splits[region]["X_train_processed"],
                client_splits[region]["y_train"],
                epochs=local_epochs,
                batch_size=batch_size
            )

            sample_counts.append(
                len(client_splits[region]["y_train"])
            )

        # Federated aggregation
        models = [
            client_models[region]
            for region in region_names.values()
        ]

        new_global_state = fedavg(
            models,
            sample_counts
        )

        global_model.load_state_dict(
            new_global_state
        )

        # Evaluate global model on every regional test set
        for region_code, region in region_names.items():

            metrics = evaluate_model(
                global_model,
                client_splits[region]["X_test_processed"],
                client_splits[region]["y_test"]
            )

            results.append({
                "round": round_number,
                "region": region,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "macro_f1": metrics["macro_f1"]
            })

            print(
                f"{region} | "
                f"Accuracy: {metrics['accuracy']:.4f} | "
                f"Macro F1: {metrics['macro_f1']:.4f}"
            )

    return global_model, results

In [90]:
# Defining the reproducible FedProx function

def run_reproducible_fedprox(
    initial_state,
    client_splits,
    region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    mu=0.01,
    seed=42
):
    # Reset random generators for reproducibility
    set_seed(seed)

    # Create the global model from the exact saved initial state
    global_model = FacilityDeliveryModel()
    global_model.load_state_dict(
        copy.deepcopy(initial_state)
    )

    results = []

    for round_number in range(1, num_rounds + 1):

        print("\n" + "=" * 50)
        print(f"Reproducible FedProx Round {round_number}")
        print("=" * 50)

        client_models = {}
        sample_counts = []

        # Create a copy of the current global model for each client
        for region_code, region in region_names.items():
            client_models[region] = copy.deepcopy(global_model)

        # Local FedProx training
        for region_code, region in region_names.items():

            print(f"\nTraining {region}...")

            train_local_fedprox(
                client_models[region],
                global_model,
                client_splits[region]["X_train_processed"],
                client_splits[region]["y_train"],
                mu=mu,
                epochs=local_epochs,
                batch_size=batch_size
            )

            sample_counts.append(
                len(client_splits[region]["y_train"])
            )

        # Federated aggregation
        models = [
            client_models[region]
            for region in region_names.values()
        ]

        new_global_state = fedavg(
            models,
            sample_counts
        )

        global_model.load_state_dict(
            new_global_state
        )

        # Evaluate global model on every regional test set
        for region_code, region in region_names.items():

            metrics = evaluate_model(
                global_model,
                client_splits[region]["X_test_processed"],
                client_splits[region]["y_test"]
            )

            results.append({
                "round": round_number,
                "region": region,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "macro_f1": metrics["macro_f1"]
            })

            print(
                f"{region} | "
                f"Accuracy: {metrics['accuracy']:.4f} | "
                f"Macro F1: {metrics['macro_f1']:.4f}"
            )

    return global_model, results

In [91]:
repro_fedavg_model, repro_fedavg_results = run_reproducible_fedavg(
    initial_state=initial_state_seed42,
    client_splits=client_splits,
    region_names=region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    seed=42
)


Reproducible FedAvg Round 1

Training North Central...
Epoch 1/1, Loss: 0.6910

Training North East...
Epoch 1/1, Loss: 0.6731

Training North West...
Epoch 1/1, Loss: 0.5898

Training South East...
Epoch 1/1, Loss: 0.6262

Training South South...
Epoch 1/1, Loss: 0.6981

Training South West...
Epoch 1/1, Loss: 0.6141
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.7400 | Macro F1: 0.4253
North West | Accuracy: 0.8431 | Macro F1: 0.4574
South East | Accuracy: 0.1923 | Macro F1: 0.1613
South South | Accuracy: 0.5122 | Macro F1: 0.3387
South West | Accuracy: 0.1863 | Macro F1: 0.1571

Reproducible FedAvg Round 2

Training North Central...
Epoch 1/1, Loss: 0.6846

Training North East...
Epoch 1/1, Loss: 0.6168

Training North West...
Epoch 1/1, Loss: 0.5238

Training South East...
Epoch 1/1, Loss: 0.6624

Training South South...
Epoch 1/1, Loss: 0.6912

Training South West...
Epoch 1/1, Loss: 0.6549
North Central | Accuracy: 0.4779 | Macro F1: 0.3233
North Ea

In [92]:
repro_fedprox_model, repro_fedprox_results = run_reproducible_fedprox(
    initial_state=initial_state_seed42,
    client_splits=client_splits,
    region_names=region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    mu=0.01,
    seed=42
)


Reproducible FedProx Round 1

Training North Central...
Epoch 1/1, Loss: 0.6915

Training North East...
Epoch 1/1, Loss: 0.6746

Training North West...
Epoch 1/1, Loss: 0.5932

Training South East...
Epoch 1/1, Loss: 0.6266

Training South South...
Epoch 1/1, Loss: 0.6982

Training South West...
Epoch 1/1, Loss: 0.6146
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.7400 | Macro F1: 0.4253
North West | Accuracy: 0.8431 | Macro F1: 0.4574
South East | Accuracy: 0.1923 | Macro F1: 0.1613
South South | Accuracy: 0.5122 | Macro F1: 0.3387
South West | Accuracy: 0.1863 | Macro F1: 0.1571

Reproducible FedProx Round 2

Training North Central...
Epoch 1/1, Loss: 0.6857

Training North East...
Epoch 1/1, Loss: 0.6191

Training North West...
Epoch 1/1, Loss: 0.5276

Training South East...
Epoch 1/1, Loss: 0.6642

Training South South...
Epoch 1/1, Loss: 0.6920

Training South West...
Epoch 1/1, Loss: 0.6566
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North 

In [93]:
#Making the local DataLoader shuffle reproducible

def train_local_reproducible(
    model,
    X_train,
    y_train,
    epochs=1,
    batch_size=32,
    seed=42
):
    dataset = FacilityDeliveryDataset(
        X_train,
        y_train
    )

    # Dedicated random generator for DataLoader shuffling
    generator = torch.Generator()
    generator.manual_seed(seed)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator
    )

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    model.train()

    for epoch in range(epochs):

        total_loss = 0.0

        for X_batch, y_batch in loader:

            optimizer.zero_grad()

            outputs = model(X_batch)

            loss = criterion(
                outputs,
                y_batch
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        average_loss = (
            total_loss / len(loader)
        )

        print(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Loss: {average_loss:.4f}"
        )

In [94]:
# FedAvg v2

def run_reproducible_fedavg_v2(
    initial_state,
    client_splits,
    region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    seed=42
):
    set_seed(seed)

    global_model = FacilityDeliveryModel()
    global_model.load_state_dict(
        copy.deepcopy(initial_state)
    )

    results = []

    for round_number in range(1, num_rounds + 1):

        print("\n" + "=" * 50)
        print(f"Reproducible FedAvg Round {round_number}")
        print("=" * 50)

        client_models = {}
        sample_counts = []

        for region_code, region in region_names.items():
            client_models[region] = copy.deepcopy(global_model)

        for region_code, region in region_names.items():

            print(f"\nTraining {region}...")

            # Deterministic but different seed for
            # each client in each round
            client_seed = (
                seed
                + round_number * 100
                + region_code
            )

            train_local_reproducible(
                client_models[region],
                client_splits[region]["X_train_processed"],
                client_splits[region]["y_train"],
                epochs=local_epochs,
                batch_size=batch_size,
                seed=client_seed
            )

            sample_counts.append(
                len(client_splits[region]["y_train"])
            )

        models = [
            client_models[region]
            for region in region_names.values()
        ]

        new_global_state = fedavg(
            models,
            sample_counts
        )

        global_model.load_state_dict(
            new_global_state
        )

        for region_code, region in region_names.items():

            metrics = evaluate_model(
                global_model,
                client_splits[region]["X_test_processed"],
                client_splits[region]["y_test"]
            )

            results.append({
                "round": round_number,
                "region": region,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "macro_f1": metrics["macro_f1"]
            })

            print(
                f"{region} | "
                f"Accuracy: {metrics['accuracy']:.4f} | "
                f"Macro F1: {metrics['macro_f1']:.4f}"
            )

    return global_model, results

In [95]:
# FedProx v2

def run_reproducible_fedprox_v2(
    initial_state,
    client_splits,
    region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    mu=0.01,
    seed=42
):
    set_seed(seed)

    global_model = FacilityDeliveryModel()
    global_model.load_state_dict(
        copy.deepcopy(initial_state)
    )

    results = []

    for round_number in range(1, num_rounds + 1):

        print("\n" + "=" * 50)
        print(f"Reproducible FedProx Round {round_number}")
        print("=" * 50)

        client_models = {}
        sample_counts = []

        for region_code, region in region_names.items():
            client_models[region] = copy.deepcopy(global_model)

        for region_code, region in region_names.items():

            print(f"\nTraining {region}...")

            # Same deterministic client/round seed scheme as FedAvg
            client_seed = (
                seed
                + round_number * 100
                + region_code
            )

            # Set the seed used by the DataLoader shuffle
            # through the local training generator
            dataset = FacilityDeliveryDataset(
                client_splits[region]["X_train_processed"],
                client_splits[region]["y_train"]
            )

            generator = torch.Generator()
            generator.manual_seed(client_seed)

            loader = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=True,
                generator=generator
            )

            criterion = nn.BCEWithLogitsLoss()

            optimizer = torch.optim.Adam(
                client_models[region].parameters(),
                lr=0.001
            )

            client_models[region].train()
            global_model.eval()

            global_params = [
                param.detach().clone()
                for param in global_model.parameters()
            ]

            for epoch in range(local_epochs):

                total_loss = 0.0

                for X_batch, y_batch in loader:

                    optimizer.zero_grad()

                    outputs = client_models[region](X_batch)

                    classification_loss = criterion(
                        outputs,
                        y_batch
                    )

                    proximal_loss = 0.0

                    for local_param, global_param in zip(
                        client_models[region].parameters(),
                        global_params
                    ):
                        proximal_loss += torch.sum(
                            (local_param - global_param) ** 2
                        )

                    proximal_loss = (
                        mu / 2
                    ) * proximal_loss

                    loss = (
                        classification_loss
                        + proximal_loss
                    )

                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item()

                average_loss = (
                    total_loss / len(loader)
                )

                print(
                    f"Epoch {epoch + 1}/{local_epochs}, "
                    f"Loss: {average_loss:.4f}"
                )

            sample_counts.append(
                len(client_splits[region]["y_train"])
            )

        models = [
            client_models[region]
            for region in region_names.values()
        ]

        new_global_state = fedavg(
            models,
            sample_counts
        )

        global_model.load_state_dict(
            new_global_state
        )

        for region_code, region in region_names.items():

            metrics = evaluate_model(
                global_model,
                client_splits[region]["X_test_processed"],
                client_splits[region]["y_test"]
            )

            results.append({
                "round": round_number,
                "region": region,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "macro_f1": metrics["macro_f1"]
            })

            print(
                f"{region} | "
                f"Accuracy: {metrics['accuracy']:.4f} | "
                f"Macro F1: {metrics['macro_f1']:.4f}"
            )

    return global_model, results

In [96]:
# Testing the seeded DataLoader

# Test deterministic shuffling

dataset = FacilityDeliveryDataset(
    client_splits["North Central"]["X_train_processed"],
    client_splits["North Central"]["y_train"]
)

generator_1 = torch.Generator()
generator_1.manual_seed(142)

loader_1 = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    generator=generator_1
)

generator_2 = torch.Generator()
generator_2.manual_seed(142)

loader_2 = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    generator=generator_2
)

batch_1_X, batch_1_y = next(iter(loader_1))
batch_2_X, batch_2_y = next(iter(loader_2))

print(
    "Same first batch:",
    torch.equal(batch_1_X, batch_2_X)
    and torch.equal(batch_1_y, batch_2_y)
)

Same first batch: True


In [97]:
repro_fedavg_model_v2, repro_fedavg_results_v2 = run_reproducible_fedavg_v2(
    initial_state=initial_state_seed42,
    client_splits=client_splits,
    region_names=region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    seed=42
)


Reproducible FedAvg Round 1

Training North Central...
Epoch 1/1, Loss: 0.6902

Training North East...
Epoch 1/1, Loss: 0.6709

Training North West...
Epoch 1/1, Loss: 0.5883

Training South East...
Epoch 1/1, Loss: 0.6242

Training South South...
Epoch 1/1, Loss: 0.6974

Training South West...
Epoch 1/1, Loss: 0.6199
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.7400 | Macro F1: 0.4253
North West | Accuracy: 0.8431 | Macro F1: 0.4574
South East | Accuracy: 0.1923 | Macro F1: 0.1613
South South | Accuracy: 0.5122 | Macro F1: 0.3387
South West | Accuracy: 0.1863 | Macro F1: 0.1571

Reproducible FedAvg Round 2

Training North Central...
Epoch 1/1, Loss: 0.6840

Training North East...
Epoch 1/1, Loss: 0.6166

Training North West...
Epoch 1/1, Loss: 0.5257

Training South East...
Epoch 1/1, Loss: 0.6718

Training South South...
Epoch 1/1, Loss: 0.6932

Training South West...
Epoch 1/1, Loss: 0.6683
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North Ea

In [98]:
repro_fedprox_model_v2, repro_fedprox_results_v2 = run_reproducible_fedprox_v2(
    initial_state=initial_state_seed42,
    client_splits=client_splits,
    region_names=region_names,
    num_rounds=5,
    local_epochs=1,
    batch_size=32,
    mu=0.01,
    seed=42
)


Reproducible FedProx Round 1

Training North Central...
Epoch 1/1, Loss: 0.6908

Training North East...
Epoch 1/1, Loss: 0.6724

Training North West...
Epoch 1/1, Loss: 0.5916

Training South East...
Epoch 1/1, Loss: 0.6246

Training South South...
Epoch 1/1, Loss: 0.6975

Training South West...
Epoch 1/1, Loss: 0.6204
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North East | Accuracy: 0.7400 | Macro F1: 0.4253
North West | Accuracy: 0.8431 | Macro F1: 0.4574
South East | Accuracy: 0.1923 | Macro F1: 0.1613
South South | Accuracy: 0.5122 | Macro F1: 0.3387
South West | Accuracy: 0.1863 | Macro F1: 0.1571

Reproducible FedProx Round 2

Training North Central...
Epoch 1/1, Loss: 0.6838

Training North East...
Epoch 1/1, Loss: 0.6185

Training North West...
Epoch 1/1, Loss: 0.5287

Training South East...
Epoch 1/1, Loss: 0.6668

Training South South...
Epoch 1/1, Loss: 0.6929

Training South West...
Epoch 1/1, Loss: 0.6644
North Central | Accuracy: 0.4792 | Macro F1: 0.3239
North 

In [99]:
print("Flower version:", fl.__version__)

print("\nNumPyClient:")
print(fl.client.NumPyClient)

print("\nFedAvg:")
print(fl.server.strategy.FedAvg)

print("\nFedProx:")
print(fl.server.strategy.FedProx)

Flower version: 1.32.1

NumPyClient:
<class 'flwr.compat.client.numpy_client.NumPyClient'>

FedAvg:
<class 'flwr.server.strategy.fedavg.FedAvg'>

FedProx:
<class 'flwr.server.strategy.fedprox.FedProx'>


In [100]:
# Building the Flower Client

class FacilityDeliveryClient(fl.client.NumPyClient):

    def __init__(
        self,
        model,
        X_train,
        y_train,
        X_test,
        y_test
    ):
        self.model = model
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test

    def get_parameters(self, config):
        return [
            val.cpu().numpy()
            for _, val in self.model.state_dict().items()
        ]

    def set_parameters(self, parameters):
        state_dict = self.model.state_dict()

        for key, value in zip(
            state_dict.keys(),
            parameters
        ):
            state_dict[key] = torch.tensor(
                value,
                dtype=state_dict[key].dtype
            )

        self.model.load_state_dict(state_dict)

    def fit(self, parameters, config):

        self.set_parameters(parameters)

        train_local(
            self.model,
            self.X_train,
            self.y_train,
            epochs=1,
            batch_size=32
        )

        return (
            self.get_parameters(config),
            len(self.y_train),
            {}
        )

    def evaluate(self, parameters, config):

        self.set_parameters(parameters)

        metrics = evaluate_model(
            self.model,
            self.X_test,
            self.y_test
        )

        loss = 0.0

        return (
            loss,
            len(self.y_test),
            {
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1": metrics["f1"],
                "macro_f1": metrics["macro_f1"]
            }
        )

In [101]:
# Test one client locally

north_central_model = FacilityDeliveryModel()

north_central_client = FacilityDeliveryClient(
    model=north_central_model,
    X_train=client_splits["North Central"]["X_train_processed"],
    y_train=client_splits["North Central"]["y_train"],
    X_test=client_splits["North Central"]["X_test_processed"],
    y_test=client_splits["North Central"]["y_test"]
)

print(type(north_central_client))
print(
    "Training samples:",
    len(north_central_client.y_train)
)
print(
    "Test samples:",
    len(north_central_client.y_test)
)

<class '__main__.FacilityDeliveryClient'>
Training samples: 3070
Test samples: 768


In [102]:
# Exchanging the parameters
"""
We want to confirm that:

1. The client can extract the model parameters into NumPy arrays.
2. Those parameters can be loaded back into the model.
3. The architecture and parameter shapes remain intact.
"""
parameters = north_central_client.get_parameters(config={})

print("Number of parameter arrays:", len(parameters))

for i, parameter in enumerate(parameters):
    print(
        f"Parameter {i}: shape={parameter.shape}"
    )

Number of parameter arrays: 6
Parameter 0: shape=(16, 5)
Parameter 1: shape=(16,)
Parameter 2: shape=(8, 16)
Parameter 3: shape=(8,)
Parameter 4: shape=(1, 8)
Parameter 5: shape=(1,)


In [103]:
# Testing local Flower

initial_parameters = north_central_client.get_parameters(
    config={}
)

updated_parameters, num_examples, metrics = (
    north_central_client.fit(
        initial_parameters,
        config={}
    )
)

print(
    "Number of returned parameter arrays:",
    len(updated_parameters)
)

print(
    "Number of training examples:",
    num_examples
)

print(
    "Metrics:",
    metrics
)

Epoch 1/1, Loss: 0.6873
Number of returned parameter arrays: 6
Number of training examples: 3070
Metrics: {}


In [111]:
def create_flower_client(context):

    # Get the client ID from Flower's Context
    client_id = int(context.node_config["partition-id"])

    # Map client ID to region
    region = list(region_names.values())[client_id]

    # Create a fresh model for this client
    model = FacilityDeliveryModel()

    # Create and return the Flower client
    client = FacilityDeliveryClient(
        model=model,
        X_train=client_splits[region]["X_train_processed"],
        y_train=client_splits[region]["y_train"],
        X_test=client_splits[region]["X_test_processed"],
        y_test=client_splits[region]["y_test"]
    )

    return client

In [114]:
for client_id in range(6):

    context = fl.common.Context(
        run_id=0,
        node_id=client_id,
        node_config={"partition-id": str(client_id)},
        state=fl.common.RecordDict(),
        run_config={}
    )

    client = create_flower_client(context)

    print(
        f"Client {client_id}: "
        f"{list(region_names.values())[client_id]} | "
        f"Train: {len(client.y_train)} | "
        f"Test: {len(client.y_test)}"
    )

Client 0: North Central | Train: 3070 | Test: 768
Client 1: North East | Train: 3598 | Test: 900
Client 2: North West | Train: 5045 | Test: 1262
Client 3: South East | Train: 1870 | Test: 468
Client 4: South South | Train: 1640 | Test: 410
Client 5: South West | Train: 1928 | Test: 483


In [115]:
fedavg_strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=6,
    min_evaluate_clients=6,
    min_available_clients=6,
)

In [107]:
# converting our saved model state to flower parameters

initial_flower_parameters = fl.common.ndarrays_to_parameters(
    [
        value.cpu().numpy()
        for value in initial_state_seed42.values()
    ]
)

print(
    "Initial Flower parameter tensors:",
    len(initial_state_seed42)
)

Initial Flower parameter tensors: 6


In [108]:
# attaching those parameters to the FedAvg strategy

fedavg_strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=6,
    min_evaluate_clients=6,
    min_available_clients=6,
    initial_parameters=initial_flower_parameters,
)

In [116]:
# test the flower simulation by running a single round

history_test = start_simulation(
    client_fn=create_flower_client,
    num_clients=6,
    config=fl.server.ServerConfig(
        num_rounds=1
    ),
    strategy=fedavg_strategy
)

print("Simulation completed.")

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=1, no round_timeout
2026-09-22 02:00:02,734	INFO worker.py:2012 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'object_store_memory': 348679372.0, 'node:127.0.0.1': 1.0, 'memory': 813585204.0, 'node:__internal_head__': 1.0, 'CPU': 4.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      No `client_resources` specified. Using minimal resources for clients.
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus'

(ClientAppActor pid=6984) Epoch 1/1, Loss: 0.5881


(ClientAppActor pid=6984) WARNING :   Deprecation Warning: The `client_fn` function must return an instance of `Client`, but an instance of `NumpyClient` was returned. Please use `NumPyClient.to_client()` method to convert it to `Client`.
(ClientAppActor pid=6984) WARNING :   Deprecation Warning: The `client_fn` function must return an instance of `Client`, but an instance of `NumpyClient` was returned. Please use `NumPyClient.to_client()` method to convert it to `Client`.


(ClientAppActor pid=6984) Epoch 1/1, Loss: 0.5861
(ClientAppActor pid=6984) Epoch 1/1, Loss: 0.6802


(ClientAppActor pid=20188) WARNING :   Deprecation Warning: The `client_fn` function must return an instance of `Client`, but an instance of `NumpyClient` was returned. Please use `NumPyClient.to_client()` method to convert it to `Client`. [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)


(ClientAppActor pid=19808) Epoch 1/1, Loss: 0.6030


INFO :      aggregate_fit: received 6 results and 0 failures
INFO :      configure_evaluate: strategy sampled 6 clients (out of 6)


(ClientAppActor pid=2876) Epoch 1/1, Loss: 0.6939


(ClientAppActor pid=2876) WARNING :   Deprecation Warning: The `client_fn` function must return an instance of `Client`, but an instance of `NumpyClient` was returned. Please use `NumPyClient.to_client()` method to convert it to `Client`. [repeated 2x across cluster]
INFO :      aggregate_evaluate: received 6 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 1 round(s) in 17.91s
INFO :      	History (loss, distributed):
INFO :      		round 1: 0.0
INFO :      


Simulation completed.


In [110]:
import sys
import importlib.util

print("Python:", sys.executable)
print("Ray installed:", importlib.util.find_spec("ray") is not None)

Python: c:\Users\usher\AppData\Local\Programs\Python\Python311\python.exe
Ray installed: True
